[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infer-actively/pymdp/blob/main/examples/advanced/infer_states_optimization/methods_test_mmp.ipynb)

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install "inferactively-pymdp" -q

In [ ]:
import numpy as np
import jax.numpy as jnp
import jax.tree_util as jtu
import jax.experimental.sparse as jsparse
from jax import nn, vmap, jit, block_until_ready
from functools import partial

from pymdp.utils import init_A_and_D_from_spec, generate_agent_specs_from_parameter_sets

# MMP setup (sequence-based inference with trivial B_dependencies)
from pymdp.utils import init_B_from_spec, get_sample_action_seq, get_sample_state, get_obs_seq_from_actions, pad_individual_Bs

# Hybrid
from pymdp.utils import apply_padding_batched
from pymdp.maths import compute_log_likelihoods_padded, deconstruct_lls
from pymdp.algos import run_mmp_hybrid # For hybrid, clustered hybrid, hybrid block and clustered hybrid block

# Clustered hybrid
from pymdp.utils import get_A_dep_clusters, apply_padding_per_cluster
from pymdp.maths import compute_log_likelihoods_per_cluster, deconstruct_log_likelihoods_per_cluster

# Hybrid block
from pymdp.utils import preprocess_A_for_block_diag, concatenate_observations_block_diag
from pymdp.maths import compute_log_likelihoods_block_diag

# Clustered hybrid block
from pymdp.utils import prep_clustered_block_data
from pymdp.maths import compute_log_likelihoods_block_diag_clustered

# End2end padded
from pymdp.utils import apply_A_end2end_padding_batched, apply_obs_end2end_padding_batched
from pymdp.maths import compute_log_likelihood_per_modality_end2end_padded
from pymdp.algos import run_mmp_end2end_padded

# Clustered end2end
from pymdp.utils import apply_A_end2end_padding_per_cluster, apply_obs_end2end_padding_per_cluster
from pymdp.maths import compute_log_likelihoods_end2end_per_cluster
from pymdp.algos import run_mmp_clustered_end2end

In [1]:
# Define coordinated parameter sets
# (num_factors, num_modalities, state_dim_upper_limit, obs_dim_upper_limit, dim_sampling_type, label)
parameter_sets = [
    (5, 5, 5, 5, 'uniform', 'low'),
    (10, 10, 10, 10, 'uniform', 'medium'),
    (25, 25, 25, 25, 'uniform', 'high'),
    # (125, 125, 125, 125, 'uniform', 'extreme'),  # Uncomment to include extreme cases
]

# Generate agent specs without dumping to file
specs = generate_agent_specs_from_parameter_sets(
    parameter_sets,
    num_agents_per_set=1,
    output_file=None  # Don't save to file
)

spec = specs['arbitrary dependencies'][1]
spec

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


{'num_factors': 10,
 'num_modalities': 10,
 'num_states': [5, 9, 5, 8, 8, 5, 2, 5, 6, 5],
 'num_obs': [2, 2, 3, 2, 2, 6, 4, 5, 6, 2],
 'A_dependencies': [[0, 4, 6, 7, 9],
  [1, 3, 8],
  [6, 9],
  [3, 8],
  [3],
  [0, 7, 9],
  [0],
  [2],
  [6],
  [5]],
 'metadata': {'num_factors': 'medium',
  'num_modalities': 'medium',
  'state_dim_upper_limit': 'medium',
  'obs_dim_upper_limit': 'medium',
  'dim_sampling_type': 'uniform'}}

In [ ]:
num_iter = 8
batch_size = 4
T = 8
tau = 1.
A_sparsity_level = None # E.g., 0.8 for 80% sparsity

num_obs = spec['num_obs']
num_states = spec['num_states']
num_controls = [2 for i in range(spec['num_factors'])]
A_dependencies = spec['A_dependencies']

# The optimized MMP routines assume trivial B_dependencies (each factor depends on its own state only)
B_dependencies = [[f] for f in range(spec['num_factors'])]

A, D = init_A_and_D_from_spec(
    num_obs,
    num_states,
    A_dependencies,
    A_sparsity_level=A_sparsity_level,
    batch_size=batch_size
)
B = init_B_from_spec(num_states, num_controls, batch_size=batch_size)

past_actions = get_sample_action_seq(num_controls, T, batch_size=batch_size)
start_state = get_sample_state(num_states, batch_size=batch_size)
obs_seq = get_obs_seq_from_actions(past_actions, start_state, A, A_dependencies, B, B_dependencies)
o_vec = [nn.one_hot(o, num_obs[m]) for m, o in enumerate(obs_seq)]

B_padded = pad_individual_Bs(B)

### Original MMP imported directly from PyMDP

In [ ]:
from pymdp.inference import update_posterior_states

infer_states_orig_pymdp = vmap(
    partial(
        update_posterior_states,
        A_dependencies=A_dependencies,
        B_dependencies=B_dependencies,
        num_iter=num_iter,
        method='mmp'
    )
)

In [2]:
qs = infer_states_orig_pymdp(A, B, o_vec, past_actions, D)
[q.shape for q in qs], qs

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array([[[0.36111808, 0.40549666, 0.02726666, 0.10550689, 0.10061175],
          [0.2484312 , 0.09603166, 0.1671715 , 0.19019508, 0.29817048],
          [0.04614736, 0.35209045, 0.14636198, 0.14591113, 0.30948907],
          [0.2627012 , 0.11562289, 0.19304022, 0.14986962, 0.27876598],
          [0.01472096, 0.29774508, 0.1697621 , 0.20376435, 0.31400752],
          [0.16501297, 0.12456366, 0.2074276 , 0.19002597, 0.31296986],
          [0.10021349, 0.14201008, 0.2368589 , 0.28314877, 0.23776871],
          [0.1478068 , 0.07672658, 0.52042156, 0.21348426, 0.04156079],
          [0.01159802, 0.2908681 , 0.3127796 , 0.07379381, 0.3109606 ]],
  
         [[0.18163696, 0.45014095, 0.24564408, 0.08549864, 0.03707937],
          [0.11695133, 0.07339869, 0.03605451, 0.3790369 , 0.39455855],
          [0.10633089, 0.14643902, 0.02627917, 0.28100353, 0.4399474 ],
 

### Hybrid method

In [ ]:
def infer_states_mmp_hybrid(obs_padded, A_padded, D, past_actions, B_padded, A_shapes, num_states, A_dependencies, B_dependencies, num_iter, tau=1.):
    lls_padded = vmap(compute_log_likelihoods_padded, in_axes=(1, None), out_axes=1)(obs_padded, A_padded)
    log_likelihoods = deconstruct_lls(lls_padded, A_shapes, has_time_axis=True)
    return vmap(
        partial(run_mmp_hybrid, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
    )(log_likelihoods, D, past_actions, B_padded)

In [ ]:
A_padded = apply_padding_batched(A)
A_shapes = [a.shape for a in A]

if A_sparsity_level is not None:
    A_padded = jsparse.BCOO.fromdense(A_padded, n_batch=1)

# obs preprocessing
obs_padded = apply_padding_batched(o_vec)

In [3]:
qs1 = infer_states_mmp_hybrid(obs_padded, A_padded, D, past_actions, B_padded, A_shapes, num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.36111808, 0.40549666, 0.02726664, 0.10550689, 0.10061175],
          [0.24843113, 0.09603172, 0.16717152, 0.19019514, 0.2981704 ],
          [0.04614739, 0.35209045, 0.14636205, 0.14591105, 0.30948907],
          [0.26270118, 0.11562287, 0.1930403 , 0.14986968, 0.27876595],
          [0.01472097, 0.29774493, 0.1697622 , 0.20376447, 0.31400737],
          [0.16501297, 0.12456361, 0.2074276 , 0.19002597, 0.31296986],
          [0.10021357, 0.14201005, 0.23685896, 0.2831487 , 0.23776865],
          [0.14780684, 0.07672661, 0.5204215 , 0.21348424, 0.0415608 ],
          [0.01159802,

In [4]:
# JIT
apply_padding_batched_jit = jit(partial(apply_padding_batched))
infer_states_mmp_hybrid_jit = jit(partial(infer_states_mmp_hybrid, A_shapes=A_shapes, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau))
obs_padded = apply_padding_batched_jit(o_vec)

qs1 = infer_states_mmp_hybrid_jit(obs_padded, A_padded, D, past_actions, B_padded)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.36111808, 0.40549666, 0.02726664, 0.10550689, 0.10061175],
          [0.24843113, 0.09603172, 0.16717152, 0.19019514, 0.2981704 ],
          [0.04614739, 0.35209045, 0.14636205, 0.14591105, 0.30948907],
          [0.26270118, 0.11562287, 0.1930403 , 0.14986968, 0.27876595],
          [0.01472097, 0.29774493, 0.1697622 , 0.20376447, 0.31400737],
          [0.16501297, 0.12456361, 0.2074276 , 0.19002597, 0.31296986],
          [0.10021357, 0.14201005, 0.23685896, 0.2831487 , 0.23776865],
          [0.14780684, 0.07672661, 0.5204215 , 0.21348424, 0.0415608 ],
          [0.01159802,

### Clustered hybrid method

In [ ]:
def infer_states_mmp_clustered_hybrid(obs_clusters, A_clusters, D, past_actions, B_padded, c2o_mapping, A_shapes, num_states, A_dependencies, B_dependencies, num_iter, tau=1.):
    ll_clusters = vmap(compute_log_likelihoods_per_cluster, in_axes=(1, None), out_axes=1)(obs_clusters, A_clusters)
    log_likelihoods = deconstruct_log_likelihoods_per_cluster(ll_clusters, A_shapes, c2o_mapping, has_time_axis=True)
    return vmap(
        partial(run_mmp_hybrid, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
    )(log_likelihoods, D, past_actions, B_padded)

In [5]:
c2o_mapping = get_A_dep_clusters(A_dependencies)
A_clusters = apply_padding_per_cluster(A, c2o_mapping)
obs_clusters = apply_padding_per_cluster(o_vec, c2o_mapping)

if A_sparsity_level is not None:
    A_clusters = [jsparse.BCOO.fromdense(a) for a in A_clusters]

qs1 = infer_states_mmp_clustered_hybrid(obs_clusters, A_clusters, D, past_actions, B_padded, c2o_mapping, A_shapes, num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.36111808, 0.40549666, 0.02726664, 0.10550689, 0.10061175],
          [0.24843113, 0.09603172, 0.16717152, 0.19019514, 0.2981704 ],
          [0.04614739, 0.35209045, 0.14636205, 0.14591105, 0.30948907],
          [0.26270118, 0.11562287, 0.1930403 , 0.14986968, 0.27876595],
          [0.01472097, 0.29774493, 0.1697622 , 0.20376447, 0.31400737],
          [0.16501297, 0.12456361, 0.2074276 , 0.19002597, 0.31296986],
          [0.10021357, 0.14201005, 0.23685896, 0.2831487 , 0.23776865],
          [0.14780684, 0.07672661, 0.5204215 , 0.21348424, 0.0415608 ],
          [0.01159802,

### Hybrid Block method

In [ ]:
def infer_states_mmp_hybrid_block(A_big, obs_big, D, past_actions, B_padded, state_shapes, cuts, num_states, A_dependencies, B_dependencies, num_iter, tau=1., use_einsum=False):
    log_likelihoods = vmap(
        partial(compute_log_likelihoods_block_diag, use_einsum=use_einsum), in_axes=(None, 1, None, None), out_axes=1
    )(A_big, obs_big, state_shapes, cuts)
    return vmap(
        partial(run_mmp_hybrid, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
    )(log_likelihoods, D, past_actions, B_padded)

In [6]:
# Create a copy with moved axes for block diagonal method (don't modify original A)
A_moveaxis = [jnp.moveaxis(a, 1, -1) for a in A]
# Preprocess A matrices for block diagonal approach
A_big, state_shapes, cuts = preprocess_A_for_block_diag(A_moveaxis)

if A_sparsity_level is not None:
    A_big = jsparse.BCOO.fromdense(A_big, n_batch=1)

obs_big = concatenate_observations_block_diag(o_vec)

qs1 = infer_states_mmp_hybrid_block(A_big, obs_big, D, past_actions, B_padded, state_shapes=state_shapes, cuts=cuts, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.36111808, 0.40549666, 0.02726664, 0.10550689, 0.10061175],
          [0.24843113, 0.09603172, 0.16717152, 0.19019514, 0.2981704 ],
          [0.04614739, 0.35209045, 0.14636205, 0.14591105, 0.30948907],
          [0.26270118, 0.11562287, 0.1930403 , 0.14986968, 0.27876595],
          [0.01472097, 0.29774493, 0.1697622 , 0.20376447, 0.31400737],
          [0.16501297, 0.12456361, 0.2074276 , 0.19002597, 0.31296986],
          [0.10021357, 0.14201005, 0.23685896, 0.2831487 , 0.23776865],
          [0.14780684, 0.07672661, 0.5204215 , 0.21348424, 0.0415608 ],
          [0.01159802,

### Clustered Hybrid Block method

In [ ]:
def infer_states_mmp_clustered_hybrid_block(A_groups, obs_groups, D, past_actions, B_padded, shape_groups, cut_groups, group_mapping, num_states, A_dependencies, B_dependencies, num_iter, tau=1.):
    num_modalities = len(A_dependencies)
    log_likelihoods = vmap(
        compute_log_likelihoods_block_diag_clustered, in_axes=(None, 1, None, None, None, None), out_axes=1
    )(A_groups, obs_groups, shape_groups, cut_groups, group_mapping, num_modalities)
    return vmap(
        partial(run_mmp_hybrid, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
    )(log_likelihoods, D, past_actions, B_padded)

In [7]:
# Cluster modalities (host-side preprocessing) and build one block-diagonal system per group
A_block, obs_block, state_shapes_block, cuts_block, group_mapping = prep_clustered_block_data(A, o_vec)

if A_sparsity_level is not None:
    A_block = [jsparse.BCOO.fromdense(a, n_batch=1) for a in A_block]

qs1 = infer_states_mmp_clustered_hybrid_block(A_block, obs_block, D, past_actions, B_padded, state_shapes_block, cuts_block, group_mapping, num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.36111808, 0.40549666, 0.02726664, 0.10550689, 0.10061175],
          [0.24843113, 0.09603172, 0.16717152, 0.19019514, 0.2981704 ],
          [0.04614739, 0.35209045, 0.14636205, 0.14591105, 0.30948907],
          [0.26270118, 0.11562287, 0.1930403 , 0.14986968, 0.27876595],
          [0.01472097, 0.29774493, 0.1697622 , 0.20376447, 0.31400737],
          [0.16501297, 0.12456361, 0.2074276 , 0.19002597, 0.31296986],
          [0.10021357, 0.14201005, 0.23685896, 0.2831487 , 0.23776865],
          [0.14780684, 0.07672661, 0.5204215 , 0.21348424, 0.0415608 ],
          [0.01159802,

### End2End padded method

In [ ]:
def infer_states_mmp_end2end_padded(obs_padded, A_padded, D, past_actions, B_padded, num_states, A_dependencies, B_dependencies, num_iter, tau=1., sparsity='ll_only'):
    lls_padded = vmap(
        partial(compute_log_likelihood_per_modality_end2end_padded, sparsity=sparsity), in_axes=(2, None)
    )(obs_padded, A_padded)
    return run_mmp_end2end_padded(lls_padded, D, past_actions, B_padded, num_states, A_dependencies, B_dependencies, num_iter=num_iter, tau=tau)

In [8]:
A_padded = apply_A_end2end_padding_batched(A)

if A_sparsity_level is not None:
    A_padded = jsparse.BCOO.fromdense(A_padded)

max_obs_dim = A_padded.shape[2]

# obs preprocessing
obs_padded = apply_obs_end2end_padding_batched(o_vec, max_obs_dim)

qs1 = infer_states_mmp_end2end_padded(obs_padded, A_padded, D, past_actions, B_padded, num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.36111835, 0.40549657, 0.02726663, 0.10550677, 0.10061172],
          [0.24843131, 0.0960317 , 0.16717157, 0.1901951 , 0.29817033],
          [0.04614734, 0.35209042, 0.14636196, 0.14591105, 0.3094892 ],
          [0.2627011 , 0.11562287, 0.1930403 , 0.14986968, 0.27876595],
          [0.01472095, 0.297745  , 0.16976199, 0.20376451, 0.31400758],
          [0.165013  , 0.12456363, 0.20742755, 0.19002609, 0.31296977],
          [0.10021351, 0.14201003, 0.23685916, 0.28314868, 0.23776864],
          [0.14780694, 0.07672665, 0.5204213 , 0.21348426, 0.04156083],
          [0.01159802,

### Clustered End2End method

In [ ]:
def infer_states_mmp_clustered_end2end(obs_clusters, A_clusters, D, past_actions, B_padded, c2o_mapping, c2s_mapping, max_state_dims, num_states, A_dependencies, B_dependencies, num_iter, tau=1., sparsity='ll_only'):
    ll_clusters = vmap(
        partial(compute_log_likelihoods_end2end_per_cluster, sparsity=sparsity), in_axes=(2, None)
    )(obs_clusters, A_clusters)
    return run_mmp_clustered_end2end(ll_clusters, D, past_actions, B_padded, c2o_mapping, c2s_mapping, max_state_dims, num_states, A_dependencies, B_dependencies, num_iter=num_iter, tau=tau)

In [9]:
A_clusters = apply_A_end2end_padding_per_cluster(A, c2o_mapping)
max_obs_dims = [a.shape[2] for a in A_clusters]
max_state_dims = [a.shape[-1] for a in A_clusters]
obs_clusters = apply_obs_end2end_padding_per_cluster(o_vec, c2o_mapping, max_obs_dims)
c2s_mapping = [[s for o in o_list for s in A_dependencies[o]] for o_list in c2o_mapping]

if A_sparsity_level is not None:
    A_clusters = [jsparse.BCOO.fromdense(a) for a in A_clusters]

qs1 = infer_states_mmp_clustered_end2end(obs_clusters, A_clusters, D, past_actions, B_padded, c2o_mapping, c2s_mapping, max_state_dims, num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.36111817, 0.40549657, 0.02726663, 0.10550692, 0.10061172],
          [0.2484312 , 0.09603175, 0.16717164, 0.1901951 , 0.29817033],
          [0.04614735, 0.3520905 , 0.146362  , 0.14591114, 0.30948895],
          [0.26270118, 0.11562286, 0.19304018, 0.14986967, 0.27876607],
          [0.01472096, 0.29774505, 0.16976209, 0.20376445, 0.3140075 ],
          [0.16501297, 0.12456366, 0.2074276 , 0.19002597, 0.31296986],
          [0.10021348, 0.14201   , 0.2368591 , 0.28314874, 0.23776868],
          [0.14780691, 0.07672664, 0.52042127, 0.21348435, 0.04156081],
          [0.01159802,